# Custom Climate Profiles Generation

#### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [12]:
import pandas as pd
import xarray as xr
import numpy as np

from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.core.data_interface import get_subsetting_options

from climakitae.explore.extreme_meteorological_year import (
    persistence_XMY,
    shock_XMY,
    persistence_get_top_hours)

from climakitae.core.constants import UNSET

from climakitae.explore.typical_meteorological_year import (
    get_cdf,
    get_cdf_monthly,
)


import warnings

warnings.filterwarnings("ignore")

### Generate an Extreme Meteorological Year (XMY) Profile

In [ ]:
start_year = 2005
end_year = 2020

In [ ]:
# Set up the TMY profile generator! The verbose option will output progress of the TMY generation
xmy = persistence_XMY(
    ## Location -- uncomment your desired option
    station_name = "Los Angeles International Airport (KLAX)",
    # latitude = latitude,
    # longitude = longitude,
    q = 0.9,
    
    # Approach -- uncomment your desired option
    warming_level = 1.5,
    # start_year = start_year,
    # end_year = end_year,
    verbose=True,
)

# Generate the profile!
xmy.generate_xmy()

**Export to non-EPW format.** TMY profiles are exported in .epw format by default, but can be exported as both `.csv` and `.tmy` file formats using the method `export_tmy_data` with the argument `extension="csv"` as shown below.

In [ ]:
xmy.export_xmy_data(extension="csv")

### Unit Testing

In [4]:
t2_ds = xr.load_dataset("hourly_air_temp_lax_2005_2020.nc")

In [5]:
t2_ds

<xarray.Dataset> Size: 3MB
Dimensions:            (sim: 4, time: 140160)
Coordinates:
  * sim                (sim) <U38 608B 'wrf_ucla_taiesm1_ssp370_r1i1p1f1' ......
    Lambert_Conformal  int32 4B 1
    lakemask           float32 4B 0.0
    landmask           float32 4B 1.0
    lat                float32 4B 33.93
    lon                float32 4B -118.4
  * time               (time) datetime64[ns] 1MB 2005-01-01 ... 2020-12-31T23...
    x                  float64 8B -4.134e+06
    y                  float64 8B 8.449e+05
Data variables:
    t2                 (sim, time) float32 2MB 13.07 12.66 12.13 ... 12.96 13.37

In [8]:
test_data = np.arange(0, 365 * 3, 1)
test_data = test_data * np.ones((2, len(test_data)))
test_ds = xr.DataArray(
    name="temperature",
    data=test_data,
    coords={
        "simulation": ["sim1", "sim2"],
        "time": pd.date_range(start="2001-01-01", end="2003-12-31"),
    },
).to_dataset()

test_ds["t2"] = (["simulation", "time"], test_data)

In [14]:
test_ds

<xarray.Dataset> Size: 44kB
Dimensions:      (simulation: 2, time: 1095)
Coordinates:
  * simulation   (simulation) <U4 32B 'sim1' 'sim2'
  * time         (time) datetime64[ns] 9kB 2001-01-01 2001-01-02 ... 2003-12-31
Data variables:
    temperature  (simulation, time) float64 18kB 0.0 1.0 ... 1.093e+03 1.094e+03
    t2           (simulation, time) float64 18kB 0.0 1.0 ... 1.093e+03 1.094e+03

In [17]:
xmy = persistence_XMY(
    station_name="Los Angeles International Airport (KLAX)",
    q=0.9,
    warming_level=1.5,
    verbose=True,
)

Initializing persistence XMY object for Los Angeles International Airport (KLAX).
Generating p90 persistence XMY.


In [ ]:
xmy.load_all_variables()

Loading data from catalog via ClimateData.
  Getting t2 (sets year range)...
2026-06-15 14:31:19 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 

Your query selected models that do not have a-priori bias adjustment. 
These models have been removed from the returned query. 
To include them, please add the following processor to your query: 
ClimateData().processes('filter_unadjusted_models': 'no')

  Loading hourly variables in parallel...
  Fetching q2 (1hr)...
  Fetching psfc (1hr)...
  Fetching u10 (1hr)...
  Fetching v10 (1hr)...
2026-06-15 14:34:07 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 

Your query selected models that do not have a-priori bias adjustment. 
These models have been removed from the returned query. 
To include them, please add the following processor to your query: 
ClimateData().processes('filter_unadjusted_models': 'no')

2026-06-15 14:34:09 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 


In [16]:
q = 0.9
result = persistence_get_top_hours(t2_ds, q, skip_last=True) 

      📊 Processing 140,160 hours (16 years) of data
      🎯 Computing 90th percentile for each hour of year


TypeError: object of type 'method' has no len()

In [10]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5]
simulations = ["sim1"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

sample_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)